In [1]:
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from cnn_base import build_t1


2026-08-30 19:53:27.248771: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-30 19:53:27.250419: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-30 19:53:27.268793: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-30 19:53:27.269282: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-30 19:53:27.678504: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [ ]:
TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
PAD_SIZE = 32
NUM_BANDS = 4
NUM_CLASSES = 6
 
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 3e-4
 
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
 
 
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]

X_train, y_train = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

print("Formato treino:", X_train.shape, y_train.shape)
print("Formato teste :", X_test.shape, y_test.shape)

In [3]:
model = build_t1()
model.summary()

optimizer = optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_test, y_test),
    shuffle=True,
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nAcurácia final no teste: {test_acc*100:.2f}% | loss: {test_loss:.4f}")

Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...
Formato treino: (324000, 32, 32, 4) (324000, 6)
Formato teste : (81000, 32, 32, 4) (81000, 6)
Model: "t1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_32x32x4 (InputLayer)  [(None, 32, 32, 4)]       0         
                                                                 
 conv1_3x3_s2 (Conv2D)       (None, 16, 16, 32)        1184      
                                                                 
 conv1_relu (Activation)     (None, 16, 16, 32)        0         
                                                                 
 conv2_1x1_s1 (Conv2D)       (None, 16, 16, 8)         264       
                                                                 
 conv2_relu (Activation)     (None, 16, 16, 8)         0         
                                                                 
 flatten (Flatten)           (Non

In [ ]:
model.save("t1_model.h5")
model.save_weights("t1_weights.h5")

with open("t1_model.json", "w") as f:
    f.write(model.to_json())

/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


: 